# Viktor prompt-complexity pipeline

This notebook builds and audits a leakage-safe training dataset from the proprietary Viktor challenge export. It does **not** embed raw data or generated outputs in the notebook.

The generated target is a transparent weak-supervision score, not measured model quality. The raw export has no final model output, usage, timing, or quality label.

## What the builder guarantees

- Every JSONL request becomes one ML example.
- Exact prefix-linked requests are reconstructed into chains.
- Exact, prefix-linked, and semantic near-duplicate task openings stay in one split.
- Normalization is fitted on training rows only; final scores retain their natural weighted distribution.
- Inputs contain only system/developer text, user text, and deterministic text features.
- Logged model and observed execution are target/audit metadata, never predictor inputs.
- Train, validation, and test files are aligned by `request_id`.

In [1]:
from pathlib import Path
import json
import statistics
import subprocess
import sys
from collections import Counter, defaultdict

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent

EXPORT_DIR = ROOT / 'export'
OUTPUT_DIR = ROOT / 'results' / 'complexity_dataset'
SEED = 42
ROOT, EXPORT_DIR, OUTPUT_DIR

(WindowsPath('c:/Users/danie/Documents/Studium/munich-ehl-2026'),
 WindowsPath('c:/Users/danie/Documents/Studium/munich-ehl-2026/export'),
 WindowsPath('c:/Users/danie/Documents/Studium/munich-ehl-2026/results/complexity_dataset'))

## 1. Build features, targets, metadata, and splits

The split is deterministic. Re-running with the same seed produces the same assignment. Near-duplicate prompt clusters are assigned atomically.

In [2]:
command = [
    sys.executable,
    str(ROOT / 'scripts' / 'build_complexity_dataset.py'),
    str(EXPORT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--seed', str(SEED),
    '--train-ratio', '0.70',
    '--validation-ratio', '0.15',
    '--test-ratio', '0.15',
]
subprocess.run(command, cwd=ROOT, check=True)

CalledProcessError: Command '['c:\\Users\\danie\\Documents\\Studium\\munich-ehl-2026\\.venv\\Scripts\\python.exe', 'c:\\Users\\danie\\Documents\\Studium\\munich-ehl-2026\\scripts\\build_complexity_dataset.py', 'c:\\Users\\danie\\Documents\\Studium\\munich-ehl-2026\\export', '--output-dir', 'c:\\Users\\danie\\Documents\\Studium\\munich-ehl-2026\\results\\complexity_dataset', '--seed', '42', '--train-ratio', '0.70', '--validation-ratio', '0.15', '--test-ratio', '0.15']' returned non-zero exit status 1.

In [ ]:
manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
summary_keys = [
    'request_count', 'trajectory_count', 'split_group_count',
    'split_request_counts', 'complexity_bands_by_split',
    'label_confidence_by_split', 'split_safety',
]
{key: manifest[key] for key in summary_keys}

## 2. Verify alignment and leakage invariants

These assertions deliberately fail if a request is duplicated, input/target order differs, or a prompt/trajectory appears in more than one split.

In [ ]:
def read_jsonl(path):
    with Path(path).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

datasets = {}
group_splits = defaultdict(set)
trajectory_splits = defaultdict(set)
all_request_ids = set()

for split in ('train', 'validation', 'test'):
    inputs = read_jsonl(OUTPUT_DIR / f'{split}_inputs.jsonl')
    targets = read_jsonl(OUTPUT_DIR / f'{split}_targets.jsonl')
    metadata = read_jsonl(OUTPUT_DIR / f'{split}_metadata.jsonl')
    input_ids = [row['request_id'] for row in inputs]
    target_ids = [row['request_id'] for row in targets]
    metadata_ids = [row['request_id'] for row in metadata]
    assert input_ids == target_ids == metadata_ids
    assert not (all_request_ids & set(input_ids))
    all_request_ids.update(input_ids)
    for row in metadata:
        group_splits[row['split_group_id']].add(split)
        trajectory_splits[row['trajectory_id']].add(split)
    datasets[split] = (inputs, targets, metadata)

assert len(all_request_ids) == manifest['request_count']
assert all(len(splits) == 1 for splits in group_splits.values())
assert all(len(splits) == 1 for splits in trajectory_splits.values())
print('Verified: aligned files, unique requests, and zero group/trajectory leakage.')

## 3. Inspect score and confidence distributions

Only scores and audit metadata are displayed here; prompts are intentionally not printed.

In [ ]:
for split, (_, targets, _) in datasets.items():
    scores = [row['complexity_score'] for row in targets]
    print({
        'split': split,
        'rows': len(scores),
        'score_min': round(min(scores), 2),
        'score_mean': round(sum(scores) / len(scores), 2),
        'score_max': round(max(scores), 2),
        'bands': dict(Counter(row['complexity_band'] for row in targets)),
        'confidence': dict(Counter(row['label_confidence'] for row in targets)),
    })

In [ ]:
# Detect provider-schema extraction failures and inspect remaining family effects.
model_scores = defaultdict(list)
nonempty_zero_intrinsic = []
for inputs, targets, metadata in datasets.values():
    for model_input, target, audit in zip(inputs, targets, metadata):
        model_scores[audit['logged_model']].append(target['complexity_score'])
        if model_input['user_prompt'].strip() and target['intrinsic_complexity'] == 0:
            nonempty_zero_intrinsic.append(target['request_id'])
assert not nonempty_zero_intrinsic
for model, scores in sorted(model_scores.items(), key=lambda item: -len(item[1])):
    print({
        'model': model, 'rows': len(scores),
        'mean_score': round(statistics.mean(scores), 2),
        'median_score': round(statistics.median(scores), 2),
    })

In [ ]:
# Optional visualization; the data pipeline itself needs no third-party packages.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Install matplotlib to display the histogram.')
else:
    plt.figure(figsize=(8, 4))
    for split, (_, targets, _) in datasets.items():
        plt.hist(
            [row['complexity_score'] for row in targets],
            bins=40, alpha=0.5, label=split,
        )
    plt.xlabel('Weak-supervision complexity score')
    plt.ylabel('Requests')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 4. Optional text-model baseline

This cell trains a simple TF–IDF ridge regressor on **user text only**. Validation is used during development; keep the test result for the final evaluation. A production router should also report downstream cost–quality performance, not only regression error.

In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import Ridge
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.pipeline import make_pipeline
except ImportError:
    print('Install scikit-learn to run the optional model baseline.')
else:
    train_inputs, train_targets, _ = datasets['train']
    val_inputs, val_targets, _ = datasets['validation']
    test_inputs, test_targets, _ = datasets['test']

    X_train = [row['user_prompt'] for row in train_inputs]
    y_train = [row['complexity_score'] for row in train_targets]
    X_val = [row['user_prompt'] for row in val_inputs]
    y_val = [row['complexity_score'] for row in val_targets]
    X_test = [row['user_prompt'] for row in test_inputs]
    y_test = [row['complexity_score'] for row in test_targets]

    model = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30_000),
        Ridge(alpha=5.0),
    )
    model.fit(X_train, y_train)

    def evaluate(name, X, y):
        prediction = model.predict(X)
        return {
            'split': name,
            'MAE': round(mean_absolute_error(y, prediction), 3),
            'RMSE': round(mean_squared_error(y, prediction) ** 0.5, 3),
            'R2': round(r2_score(y, prediction), 3),
        }

    print(evaluate('validation', X_val, y_val))
    print(evaluate('test', X_test, y_test))

## 5. Experimental model catalog — unverified assumptions only

Sections 1–4 produce a 0–100 complexity score per request. A router needs the other half of
the mapping: for each candidate model, the complexity above which it should not be trusted.
`scripts/model_catalog.py` demonstrates how assumed benchmark results could be mapped to
cutoffs. Its bundled cards and tier order are not verified evidence.

**The model ids are anonymized.** The bundled identity mapping, benchmark values, sources,
tier order, and `scripts/pricing.json` are assumptions unless the organizers explicitly
verify them. Do not use this section as submission evidence without approved replacements.

The chain is kept short so each link can be argued with:

| step | what it does |
|---|---|
| benchmark panel | five agentic benchmarks, weighted by how well each proxies *this* workload |
| additive fit | joint model-ability / benchmark-difficulty estimate, on the logit scale |
| spread | rescale fitted ability across the catalog to 0–1 |
| quantile | map onto `[quantile_floor, 1]`, minus a margin that grows as evidence thins |
| cutoff | read that quantile off the **observed training** complexity distribution |

A benchmark score says nothing about this dataset's 0–100 scale, so the fit is used only to
**order and space** the models; the thresholds themselves come from the empirical
distribution. Calibration uses the training split only — calibrating on validation or test
would leak the evaluation set into the router, the invariant sections 1–2 exist to protect.

In [ ]:
import importlib
import sys

if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))
import model_catalog

importlib.reload(model_catalog)

train_scores = [row['complexity_score'] for row in datasets['train'][1]]
catalog = model_catalog.build_catalog(train_scores, risk_aversion=0.10)

print(f"calibrated on {catalog['calibrated_on']['n_requests']} training requests "
      f"(scores {catalog['calibrated_on']['score_min']}–{catalog['calibrated_on']['score_max']})")
print(f"frontier / routing fallback: {catalog['frontier_model']}")
print(f"detectability band: {catalog['tie_band']:.2f} score points\n")

header = (f"{'model':<20}{'tier':>5}{'fitted':>9}{'evidence':>10}{'benchmarks':>12}"
          f"{'cutoff':>8}   indistinguishable from")
print(header)
print('-' * (len(header) + 20))
for entry in catalog['models']:
    peers = ', '.join(entry['indistinguishable_from']) or '—'
    flag = '*' if entry['is_frontier'] else ('~' if entry['imputed'] else ' ')
    print(f"{entry['model_id']:<20}{entry['tier_group']:>5}"
          f"{entry['expected_score']:>9.2f}{entry['evidence_coverage']:>9.0%}"
          f"{entry['n_benchmarks']:>12}{entry['complexity_cutoff']:>8.2f} {flag} {peers}")
print('\n* frontier (used when nothing else covers a request)   ~ imputed, no public evidence')
print('evidence = share of the weighted benchmark panel this model actually has results for')
print('Selection is capability-only. Prices are recorded on each model card but are never')
print('consulted when ranking or routing, so no cost objective enters this evaluation.')

### 5.1 Which capability differences are real

Coverage in the bundled assumption panel is ragged — Opus 5 has no assumed SWE-bench Pro
result, the GPT-5.6 tiers have no SWE-bench Verified, `claude-opus-4-6` has nothing at all —
and the benchmarks also **disagree systematically about which family is stronger**. Those are
two different problems and they need two different fixes.

*Ragged coverage.* A naive composite (`score / best observed score`, averaged over whatever a
model happens to have) compares models on incomparable panels, so being measured only where
your family is strong is rewarded. Fix: compare every pair of models **only on the benchmarks
both were measured on**, then reconcile those pairwise differences into one ability by
weighted least squares.

*Systematic disagreement.* Least squares is still a global fit, so a model can be dragged down
through third parties. This is not hypothetical — it produced a live bug in an earlier version
of this notebook:

> `gpt-5.6-terra` was ranked **above** `gpt-5.6-sol`, even though Sol beats Terra on all three
> benchmarks they share (88.8 vs 87.4, 80.0 vs 77.4, 53.6 vs 50.4). The cause: Sol is unusually
> weak on SWE-bench Pro — 12 points below its own fit, the largest residual in the panel — and
> Terra was never measured on it. The penalty reached Terra's rival through models Terra never
> met.

Fix: **direct evidence outranks indirect inference.** Abilities are projected onto the partial
order of head-to-head dominance — if A beat B on every benchmark they share, A may not rank
below B — pooling any violating pair until none remain. Pooling equalises rather than
reorders, which is the smallest correction that removes the false claim.

Two things worth checking below: that the fitted difficulties rank **SWE-bench Verified as the
easiest** benchmark (everyone scores 88–97 on it) and **Agents' Last Exam as the hardest**; and
the **detectability band** from the fit's own residuals — the smallest gap in fitted score
worth believing.

In [ ]:
cards = model_catalog._catalog_cards()
fit = model_catalog.fit_capability(cards)

print('fitted benchmark difficulty (logit; higher = everyone scores well = easier):')
for name, value in sorted(fit['difficulty'].items(), key=lambda kv: -kv[1]):
    print(f"  {model_catalog.BENCHMARKS[name]['label']:<24}{value:+7.3f}"
          f"   weight {model_catalog.BENCHMARKS[name]['weight']:.2f}")

print('\nhead-to-head dominance (A scored higher on every benchmark A and B both have):')
for winner, loser, shared in model_catalog.dominance_pairs(cards):
    print(f"  {winner:<20} > {loser:<20} on {shared} shared benchmarks")

repairs = fit['dominance_repaired']
print(f"\ndominance repairs applied: {len(repairs)}")
for repair in repairs:
    print(f"  {repair['winner']} had been ranked below {repair['loser']}"
          f"  ->  pooled {', '.join(repair['pooled_members'])}")
violations = [(w, l) for w, l, _ in model_catalog.dominance_pairs(cards)
              if fit['models'][w]['ability'] < fit['models'][l]['ability'] - 1e-12]
print(f"violations remaining after repair: {len(violations)}")

print('\nnaive composite vs fitted ability (rank shown as naive -> fitted):')
naive = model_catalog.naive_composite(cards)
naive_rank = {m: i + 1 for i, m in enumerate(
    sorted((k for k, v in naive.items() if v is not None), key=lambda k: -naive[k]))}
fit_rank = {m['model_id']: i + 1 for i, m in enumerate(catalog['models'])}

print(f"  {'model':<20}{'naive':>8}{'fitted':>8}{'rank':>10}{'benchmarks':>12}")
for entry in catalog['models']:
    mid = entry['model_id']
    value = naive.get(mid)
    moved = f"{naive_rank.get(mid, '-')} -> {fit_rank[mid]}"
    print(f"  {mid:<20}{'n/a' if value is None else f'{value:.3f}':>8}"
          f"{entry['expected_score']:>8.2f}{moved:>10}{entry['n_benchmarks']:>12}")

print(f"\nDetectability band: {catalog['tie_band']:.2f} score points.")
for tier in sorted({e['tier_group'] for e in catalog['models']}):
    members = [e for e in catalog['models'] if e['tier_group'] == tier]
    span = max(m['expected_score'] for m in members) - min(m['expected_score'] for m in members)
    print(f"  tier {tier}: {', '.join(m['model_id'] for m in members)}")
    print(f"          fitted spread {span:.2f} pts "
          f"| members are interchangeable for routing")

### 5.2 Cutoffs against the distribution they were calibrated on

The cutoffs only mean something relative to the complexity distribution they were read off.
Plotting them together shows how much of the observed workload each model is trusted with —
and makes it obvious when a cutoff sits somewhere the distribution is thin, where small
changes in the fit move a lot of traffic.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Install matplotlib to display the cutoff chart.')
    for entry in catalog['models']:
        share = sum(1 for s in train_scores if s <= entry['complexity_cutoff']) / len(train_scores)
        print(f"  {entry['model_id']:<20} cutoff {entry['complexity_cutoff']:>6.2f}"
              f"   trusted with {share:>6.1%} of training requests")
else:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.hist(train_scores, bins=45, color='0.88', edgecolor='0.65', label='training requests')
    top = ax.get_ylim()[1]
    ax.set_ylim(0, top * 1.05)
    colours = plt.cm.viridis(
        [i / max(len(catalog['models']) - 1, 1) for i in range(len(catalog['models']))])

    for position, (colour, entry) in enumerate(zip(colours, catalog['models'])):
        share = sum(1 for s in train_scores
                    if s <= entry['complexity_cutoff']) / len(train_scores)
        ax.axvline(entry['complexity_cutoff'], color=colour, linewidth=1.8)
        ax.annotate(
            f"{entry['model_id']}  cutoff {entry['complexity_cutoff']:.1f}  ({share:.0%})",
            xy=(entry['complexity_cutoff'], top),
            xytext=(3, -11 - 12 * position), textcoords='offset points',
            fontsize=8, color=colour, va='top',
        )

    ax.set_xlabel('Weak-supervision complexity score (training split)')
    ax.set_ylabel('Requests')
    ax.set_title('Benchmark-derived quality cutoffs over the observed complexity distribution')
    plt.tight_layout()
    plt.show()

    print('Percentage after each label = share of training requests at or below that cutoff.')

### 5.3 The logged policy is not complexity-aware — which is where the headroom is

Before trusting any cutoff, it is worth asking what the *logged* assignment already does. If
Viktor were routing by difficulty, hard requests would sit on strong models and the correlation
between a request's complexity and its serving model's fitted capability would be clearly
positive.

It is not. That single fact carries most of the routing argument in this notebook, and it cuts
two ways:

- **In favour of a router:** there is real headroom, because the current assignment leaves
  complexity on the table. Any complexity-aware policy is competing against a policy that
  ignores complexity.
- **Against over-claiming:** because the logged data contains almost no complexity-driven
  model assignment, it also contains almost no evidence about which model a hard request
  *needs*. The observational log can tell us the current policy is undifferentiated; it cannot
  tell us the cutoffs are correct.

In [ ]:
import statistics


def pearson(xs, ys):
    mx, my = statistics.mean(xs), statistics.mean(ys)
    numerator = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    denominator = (sum((x - mx) ** 2 for x in xs) * sum((y - my) ** 2 for y in ys)) ** 0.5
    return numerator / denominator if denominator else 0.0


capability = {m['model_id']: m['expected_score'] for m in catalog['models']}

scores = [t['complexity_score'] for t in datasets['train'][1]]
served = [m['logged_model'] for m in datasets['train'][2]]

print(f"corr(complexity, fitted capability of the model that served it) = "
      f"{pearson(scores, [capability[m] for m in served]):+.3f}")

grouped = {}
for score, model in zip(scores, served):
    grouped.setdefault(model, []).append(score)

between = statistics.pstdev([statistics.mean(v) for v in grouped.values()])
within = statistics.mean([statistics.pstdev(v) for v in grouped.values() if len(v) > 1])
print(f"\nspread of per-model mean complexity : {between:>5.1f}")
print(f"typical spread *within* one model    : {within:>5.1f}")
print(f"overall complexity range             : {min(scores):.1f}–{max(scores):.1f}")
print('\nWithin-model variation swamps between-model variation: whatever decided which')
print('model served a request, it was almost entirely something other than difficulty.')

### 5.4 Sweeping risk aversion

`risk_aversion` is the single knob on the catalog. It is the maximum quantile a model can be
demoted by: `0.0` trusts the benchmarks completely, higher values pull every cutoff down so
more work escalates to stronger models.

Routing is done **per trajectory, on its hardest call**. That respects the export's
one-model-per-trajectory premise and avoids the cache reset `scripts/cost_model.py` charges for
a mid-trajectory switch.

**This evaluation is capability-only — no cost term appears in it.** The router picks the
weakest model that still covers a request, resolved at tier granularity so it never prefers a
model the benchmarks cannot actually distinguish from a better-evidenced one. What risk
aversion buys is margin against a mis-estimated cutoff; what it costs is **headroom** —
capability provisioned above what the trajectory needed. That trade is the whole point of the
sweep, and `headroom` is the column to read.

`weaker` / `stronger` compare the routed model against the one that actually served the
trajectory, by fitted ability. Because the logged policy is close to complexity-blind (§5.3), a
high `weaker` share is not by itself evidence of a good router — it mostly reflects that the
logged assignment was not chosen on difficulty in the first place.

In [ ]:
from collections import defaultdict

train_inputs, train_targets, train_meta = datasets['train']
by_trajectory = defaultdict(list)
for inp, tgt, meta in zip(train_inputs, train_targets, train_meta):
    by_trajectory[meta['trajectory_id']].append(
        (tgt['complexity_score'], meta['logged_model']))

CLAUDE_ONLY = [m['model_id'] for m in catalog['models'] if m['family'] == 'claude']
ability = {m['model_id']: m['expected_score'] for m in catalog['models']}


def sweep(allowed=None, snap=False):
    rows = []
    for risk in (0.0, 0.05, 0.10, 0.20, 0.30):
        swept = model_catalog.build_catalog(train_scores, risk_aversion=risk,
                                            snap_to_band=snap)
        mix, fallbacks = defaultdict(int), 0
        headroom, weaker, stronger = [], 0, 0
        for calls in by_trajectory.values():
            hardest = max(c[0] for c in calls)
            choice, reason = model_catalog.route_trajectory(
                [c[0] for c in calls], swept, allowed=allowed)
            mix[choice] += 1
            fallbacks += reason == 'fallback_frontier'
            logged = calls[0][1]
            headroom.append(
                next(m['complexity_cutoff'] for m in swept['models']
                     if m['model_id'] == choice) - hardest)
            if ability[choice] < ability[logged] - 1e-9:
                weaker += 1
            elif ability[choice] > ability[logged] + 1e-9:
                stronger += 1
        rows.append((risk, sum(headroom) / len(headroom),
                     weaker / len(by_trajectory), stronger / len(by_trajectory),
                     fallbacks / len(by_trajectory), mix))
    return rows


def show(title, rows):
    print(f'\n{title}')
    print(f"{'risk':>6}{'headroom':>10}{'weaker':>9}{'stronger':>10}{'fallback':>10}"
          f"   route mix (trajectories)")
    print('-' * 104)
    for risk, headroom, weaker, stronger, fallback, mix in rows:
        share = ', '.join(f'{m} {n}/{len(by_trajectory)}'
                          for m, n in sorted(mix.items(), key=lambda kv: -kv[1]))
        print(f'{risk:>6.2f}{headroom:>10.1f}{weaker:>8.0%}{stronger:>10.0%}'
              f'{fallback:>10.0%}   {share}')


logged_claude = sum(1 for c in by_trajectory.values() if c[0][1].startswith('claude'))
print(f'{len(by_trajectory)} training trajectories; '
      f'logged mix is {logged_claude}/{len(by_trajectory)} Claude')

show('All nine catalogued models available:', sweep())
show(f'Claude family only ({len(CLAUDE_ONLY)} models):', sweep(CLAUDE_ONLY))
show('Claude family only, indistinguishable models pooled (snap_to_band=True):',
     sweep(CLAUDE_ONLY, snap=True))

print()
print('headroom = mean gap between the chosen cutoff and the trajectory hardest call;')
print('           lower means less over-provisioned intelligence, negative is impossible')
print('           because a covering model is always chosen when one exists')
print('weaker / stronger = share routed to a model of lower / higher fitted ability')
print('           than the one that actually served it')
print('fallback = trajectories above every cutoff, escalated to the frontier model')
print()
print('No cost term appears anywhere above. Raising risk aversion buys margin against')
print('a mis-estimated cutoff and pays for it in over-provisioned capability, which is')
print('what the headroom column measures.')

### 5.5 What this catalog gets wrong

The catalog is deliberately auditable, which means its assumptions are visible rather than
absent. Four are load-bearing enough to state before anything else.

**The benchmarks cannot resolve the top of the catalog.** Luna, Terra, Sol, Fable 5 and Opus 5
all fall inside one detectability band — a 2.6-point spread. Everything in that band is
interchangeable as far as this evidence goes, and the catalog treats it that way rather than
inventing an ordering. If a choice has to be made among them, it has to be made on grounds this
catalog does not supply.

**A single capability number is a simplification the data actively resists.** The benchmarks
disagree about which family is stronger, and not randomly: Sol is 12 points below its own fit
on SWE-bench Pro while Fable 5 is 8 points below its own fit on Agents' Last Exam. Those are
model×benchmark interactions, and collapsing them into one scalar hides them by construction.
The dominance repair in §5.1 stops that from producing an outright false ordering; it does not
make the scalar true.

**Thin evidence is not the same as low capability, and the two are easy to confuse.** A model
with few assumed observations gets a wider safety margin and therefore a lower cutoff, which under
a weakest-sufficient rule would quietly attract *more* traffic. `claude-opus-4-6` — imputed,
with no public evidence at all — took 205 of 700 trajectories before selection was made
tier-aware. It now defers to its better-corroborated tier peer, but the general hazard remains:
poorly measured models look artificially safe to route to.

**A cutoff assumes failures are the hardest tasks.** Mapping a benchmark success rate onto a
complexity ceiling only works if a model fails the hard things first. Models fail
idiosyncratically, so a cutoff is a central tendency, not a guarantee.

> **Scope note.** Nothing here optimises or evaluates cost — selection is capability-only by
> design. That means this section supplies the *quality* axis of the cost–quality frontier the
> challenge asks for, and not the frontier itself; the cost axis has to come from
> `scripts/cost_model.py`.

In [ ]:
print('Recorded limitations (scripts/model_catalog.py):\n')
for i, item in enumerate(catalog['limitations'], 1):
    print(f"{i:>2}. {item}\n")

print('Benchmarks deliberately excluded from the panel:\n')
for name, reason in catalog['excluded_benchmarks'].items():
    print(f"  - {name}: {reason}\n")

worst = max(catalog['models'], key=lambda m: m['max_source_disagreement'])
top_tier = [m['model_id'] for m in catalog['models'] if m['tier_group'] == 1]
print(f"Largest single-model source disagreement: {worst['model_id']} spans "
      f"{worst['max_source_disagreement']:.1f} benchmark points across assumed sources —\n"
      f"wider than the {catalog['tie_band']:.2f}-point band that holds the top "
      f"{len(top_tier)} models ({', '.join(top_tier)})\nin a single indistinguishable tier.")

## Files produced

For each split, `*_inputs.jsonl` contains predictor-safe text/features, `*_targets.jsonl` contains the score and its components, and `*_metadata.jsonl` contains audit-only identifiers and the logged model. `all.jsonl` combines them for analysis, while `manifest.json` records formulas, scaling caps, split counts, and limitations.

Before using this score as a router target, manually review a stratified sample from the low/medium/high bands. The score is deliberately auditable, but its weights still encode assumptions that should be validated against human or judge-model ratings.